In [26]:
import json
import os

In [27]:
evaluations_folder = "evaluations/coqstoq-results"
baseline_folder = "eval-baseline"
unseen_folder = "eval-grpo-unseen"

In [29]:
def find_json_files(folder_path):
    json_file_paths = []
    if not os.path.isdir(folder_path):
        print(f"Error: '{folder_path}' is not a valid directory.")
        return json_file_paths

    for root, _, files in os.walk(folder_path):
        for file in files:
            if file.endswith(".json"):
                json_file_paths.append(os.path.join(root, file))
    return json_file_paths

In [30]:
baseline_files = find_json_files(f"{evaluations_folder}/{baseline_folder}")
unseen_files = find_json_files(f"{evaluations_folder}/{unseen_folder}")


baseline_files_filtered = ["/".join(file.split("/")[3:]) for file in baseline_files]
unseen_files_filtered = ["/".join(file.split("/")[3:]) for file in unseen_files]

common_files = set(baseline_files_filtered).intersection(set(unseen_files_filtered))
common_files = list(common_files)
print(common_files[0])
print(len(common_files))

compcert/backend/ValueDomain.v/3018-0.json
360


In [31]:
baseline_path = "evaluations/coqstoq-results/eval-baseline"
unseen_path = "evaluations/coqstoq-results/eval-grpo-unseen"
filtered_files = []
for file in common_files:
    with open(f"{baseline_path}/{file}", "r") as f:
        # Check if the file is empty
        if os.path.getsize(f"{baseline_path}/{file}") == 0:
            continue
        data_rango = json.load(f)
    with open(f"{unseen_path}/{file}", "r") as f:
        # Check if the file is empty
        if os.path.getsize(f"{unseen_path}/{file}") == 0:
            continue
        data_proofmind = json.load(f)
    if data_rango["time"] == None or data_proofmind["time"] == None:
        continue
    filtered_files.append(file) 

print(len(filtered_files))
        

355


In [32]:
from statistics import mean, median, mode, stdev, variance

##### RANGO

In [33]:
correct_counter = 0 

total = 0
average_time = 0
valid_proof_counter = 0
times_rango = []
num_attempts_rango = []
for file in filtered_files:
    with open(f"{baseline_path}/{file}", "r") as f:
        data = json.load(f)
    #print(data)
    if data["proof"] != None:
        correct_counter += 1
        average_time += data["time"]
        valid_proof_counter += 1
        times_rango.append(data["time"])
        num_attempts_rango.append(data["n_attempts"])

print("Max number of attempts: ", max(num_attempts_rango))
print("Median number of attempts: ", median(num_attempts_rango))
print("Average number of attempts: ", mean(num_attempts_rango))
print("Max time: ", max(times_rango))
print("Median time: ", median(times_rango))
print(f"Average time: {average_time/valid_proof_counter}")
print(f"Correct: {correct_counter/len(filtered_files)*100}")

print(correct_counter)
print(len(filtered_files))

Max number of attempts:  134
Median number of attempts:  4
Average number of attempts:  15.557522123893806
Max time:  474.324152469635
Median time:  23.97709321975708
Average time: 72.75340665336203
Correct: 31.83098591549296
113
355


##### ProofMindRL

In [34]:
correct_counter = 0 
total = 0
unseen_path = "evaluations/coqstoq-results/eval-grpo-unseen"
average_time = 0
valid_proof_counter = 0
times_proofmind = []
num_attempts_proofmind = []
for file in common_files:
    with open(f"{unseen_path}/{file}", "r") as f:
        data = json.load(f)
    #print(data)
    if data["proof"] != None:
        correct_counter += 1
    
        average_time += data["time"]
        valid_proof_counter += 1
        times_proofmind.append(data["time"])
        num_attempts_proofmind.append(data["n_attempts"])

print("Max number of attempts: ", max(num_attempts_proofmind))
print("Median number of attempts: ", median(num_attempts_proofmind))
print("Average number of attempts: ", mean(num_attempts_proofmind))
print("Max time: ", max(times_proofmind))
print("Median time: ", median(times_proofmind))
print(f"Average time: {average_time/valid_proof_counter}")
print(f"Correct: {correct_counter/len(filtered_files)*100}")
print(correct_counter)

Max number of attempts:  217
Median number of attempts:  4
Average number of attempts:  17.26050420168067
Max time:  553.1180534362793
Median time:  31.246729612350464
Average time: 77.67309718973496
Correct: 33.52112676056338
119


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

def plot_time_distribution(times, title="Distribution of Proof Times"):
    """
    Plot the distribution of proof times using both a histogram and a box plot.
    
    Args:
        times (list): List of time values
        title (str): Title for the plot
    """
    # Create a figure with two subplots
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8))
    
    # Plot histogram with KDE
    sns.histplot(data=times, kde=True, ax=ax1)
    ax1.set_title(f"{title} - Histogram")
    ax1.set_xlabel("Time (seconds)")
    ax1.set_ylabel("Count")
    
    # Plot box plot
    sns.boxplot(x=times, ax=ax2)
    ax2.set_title(f"{title} - Box Plot")
    ax2.set_xlabel("Time (seconds)")
    
    # Adjust layout to prevent overlap
    plt.tight_layout()

plot_time_distribution(num_attempts_rango, "rango")

In [ ]:
plot_time_distribution(num_attempts_proofmind, "proofmind")

In [33]:
def compare_proofs(filtered_files, proofmind_path, goback_path):
    """
    Compare proof values between ProofMind and ProofMind Go-Back-N results.
    
    Args:
        filtered_files (list): List of file paths to compare
        proofmind_path (str): Path to ProofMind results
        goback_path (str): Path to ProofMind Go-Back-N results
    
    Returns:
        dict: Dictionary containing comparison statistics
    """
    comparison_stats = {
        'both_success': 0,
        'both_failure': 0,
        'only_proofmind': 0,
        'only_goback': 0,
        'total_files': len(filtered_files)
    }
    
    for file in filtered_files:
        # Read ProofMind result
        with open(f"{proofmind_path}/{file}", "r") as f:
            proofmind_data = json.load(f)
        
        # Read Go-Back-N result
        with open(f"{goback_path}/{file}", "r") as f:
            goback_data = json.load(f)
        
        # Compare proof values
        proofmind_success = proofmind_data["proof"] is not None
        goback_success = goback_data["proof"] is not None
        


        if proofmind_success and goback_success:
            comparison_stats['both_success'] += 1
        elif not proofmind_success and not goback_success:
            comparison_stats['both_failure'] += 1
        elif proofmind_success and not goback_success:
            comparison_stats['only_proofmind'] += 1
            #print(proofmind_data["proof"])
            #print("----------------")
        elif not proofmind_success and goback_success:
            #print(goback_data["proof"])
            #print("-------------")
            comparison_stats['only_goback'] += 1
    
    # Calculate percentages
    total = comparison_stats['total_files']
    comparison_stats['both_success_pct'] = (comparison_stats['both_success'] / total) * 100
    comparison_stats['both_failure_pct'] = (comparison_stats['both_failure'] / total) * 100
    comparison_stats['only_proofmind_pct'] = (comparison_stats['only_proofmind'] / total) * 100
    comparison_stats['only_goback_pct'] = (comparison_stats['only_goback'] / total) * 100
    
    return comparison_stats

# Example usage:
proofmind_path = "evaluations/coqstoq-results/test-proofmindrl"
rango_path = "evaluations/coqstoq-results/test-rango"

stats = compare_proofs(filtered_files, proofmind_path, rango_path)

# Print results
print(f"Total files analyzed: {stats['total_files']}")
print(f"Both successful: {stats['both_success']} ({stats['both_success_pct']:.2f}%)")
print(f"Both failed: {stats['both_failure']} ({stats['both_failure_pct']:.2f}%)")
print(f"Only proofmind successful: {stats['only_proofmind']} ({stats['only_proofmind_pct']:.2f}%)")
print(f"Only Rango successful: {stats['only_goback']} ({stats['only_goback_pct']:.2f}%)")

Total files analyzed: 316
Both successful: 81 (25.63%)
Both failed: 207 (65.51%)
Only proofmind successful: 12 (3.80%)
Only Rango successful: 16 (5.06%)
